In [6]:
import os
import pandas as pd
import numpy as np
from datasets import Dataset
from setfit import SetFitModel, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

os.environ["WANDB_DISABLED"] = "true"

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"✅ CUDA disponible. Nombre de GPU : {torch.cuda.device_count()}")
    print(f"Nom du GPU : {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
else:
    print("❌ Aucun GPU détecté, utilisation du CPU.")
    device = torch.device("cpu")

✅ CUDA disponible. Nombre de GPU : 1
Nom du GPU : NVIDIA GeForce RTX 4070 Ti


In [ ]:
df = pd.read_csv('../data/fake_news_detection.csv', index_col=None)

df = df.rename(columns={'tweet': 'text', 'is_real': 'label'})
df['label'] = df['label'].map({'real': 1, 'fake': 0})

global_train_df, test_df = train_test_split(
    df, 
    test_size=0.20, 
    random_state=42, 
    stratify=df['label']
)


test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

print(f"Taille de la base d'entraînement globale (80%) : {len(global_train_df)}")
print(f"Taille de la base de test finale fixe (20%) : {len(test_dataset)}")

Taille de la base d'entraînement globale (80%) : 800
Taille de la base de test finale fixe (20%) : 200


In [ ]:
def compute_metrics(y_pred, y_test):
    accuracy = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="binary"
    )
    return {
        "accuracy":  round(accuracy, 4),
        "precision": round(precision, 4),
        "recall":    round(recall, 4),
        "f1":        round(f1, 4),
    }

In [13]:
sample_sizes = [8, 10, 20, 50, 100]
epochs_list = [1, 5, 10]

all_results = []

for num_samples in sample_sizes:
    for num_epochs in epochs_list:
        print("\n" + "="*50)
        print(f"EXPÉRIENCE : {num_samples} échantillons | {num_epochs} époques")
        print("="*50)

        n_per_class = num_samples // 2
        train_sub_0 = global_train_df[global_train_df['label'] == 0].sample(n=n_per_class, random_state=42)
        train_sub_1 = global_train_df[global_train_df['label'] == 1].sample(n=n_per_class, random_state=42)

        train_df_raw = pd.concat([train_sub_0, train_sub_1]).sample(frac=1, random_state=42)

        eval_df = global_train_df.drop(index=train_df_raw.index)

        train_df = train_df_raw.reset_index(drop=True)
        eval_df = eval_df.reset_index(drop=True)

        current_train_dataset = Dataset.from_pandas(train_df)
        current_eval_dataset = Dataset.from_pandas(eval_df.reset_index(drop=True))

        model = SetFitModel.from_pretrained("sentence-transformers/paraphrase-mpnet-base-v2",
                                            device="cuda"
                                            )

        args = TrainingArguments(
            batch_size=16,
            num_epochs=num_epochs,
            eval_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True
        )
        
        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=current_train_dataset,
            eval_dataset=current_eval_dataset,
            metric=compute_metrics,
        )

        trainer.train()

        print(f"Évaluation sur le test set de l'expérience ({num_samples} samples, {num_epochs} epochs)...")
        test_metrics = trainer.evaluate(test_dataset)

        result_entry = {
            "training_samples": num_samples,
            "epochs": num_epochs,
            "accuracy": test_metrics["accuracy"],
            "precision": test_metrics["precision"],
            "recall": test_metrics["recall"],
            "f1_score": test_metrics["f1"]
        }
        all_results.append(result_entry)
        print(f"Résultats obtenus : {test_metrics}")


EXPÉRIENCE : 8 échantillons | 1 époques


model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Map: 100%|██████████| 8/8 [00:00<00:00, 3255.82 examples/s]
***** Running training *****
  Num unique pairs = 40
  Batch size = 16
  Num epochs = 1


Epoch,Training Loss,Validation Loss
1,0.198000,0.180621


***** Running evaluation *****


Évaluation sur le test set de l'expérience (8 samples, 1 epochs)...
Résultats obtenus : {'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}

EXPÉRIENCE : 8 échantillons | 5 époques


model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Map: 100%|██████████| 8/8 [00:00<00:00, 3489.44 examples/s]
***** Running training *****
  Num unique pairs = 40
  Batch size = 16
  Num epochs = 5


Epoch,Training Loss,Validation Loss
1,0.198000,0.180147
2,0.198000,0.109706


KeyboardInterrupt: 

In [ ]:
# Conversion des résultats en DataFrame pour une lecture claire sous forme de tableau
df_results = pd.DataFrame(all_results)

print("\n" + "#"*50)
print("     RAPPORT DE PERFORMANCE GLOBAL (SUR LE TEST SET)")
print("#"*50)

# Trier par F1-score pour voir rapidement la meilleure configuration
df_results = df_results.sort_values(by="f1_score", ascending=False).reset_index(drop=True)
df_results